# SynthScope quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SMSD75/SynthScope/blob/main/notebooks/synthscope_quickstart.ipynb)

Run state-of-the-art **AI-generated image & video detectors** in a few lines with the [`synthscope`](https://github.com/SMSD75/SynthScope) library.

> Tip: switch the runtime to a **GPU** (Runtime → Change runtime type) — the backbones are large.

## 1. Install

Installs `synthscope` (which pulls PyTorch, transformers, etc.) and OpenAI CLIP (needed by a couple of detectors). Takes a few minutes on first run.

In [ ]:
%pip install -q "git+https://github.com/SMSD75/SynthScope.git" "git+https://github.com/openai/CLIP.git"

## 2. Get detector weights

This notebook uses **CuRe**, whose authors publish their weights on GitHub (a 25 MB adapter). The first prediction also downloads the frozen PE-Core backbone (2.7 GB) from Hugging Face. Other detectors need their weights as `<key>.pth` in the same folder; each model's README links the official download.

In [ ]:
import os, urllib.request

CKPT = "/content/synthscope_ckpts" if os.path.isdir("/content") else os.path.expanduser("~/synthscope_ckpts")
os.makedirs(CKPT, exist_ok=True)
os.environ["SYNTHSCOPE_CKPT_DIR"] = CKPT          # must be set before importing synthscope

if not os.path.exists(f"{CKPT}/cure.pth"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/manic-cui/CuRe/main/weights/cure_adapter.pt", f"{CKPT}/cure.pth")

import synthscope
[m["key"] for m in synthscope.list_models() if m["available"]]   # detectors that can run here

## 3. Upload an image (or video)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    path = next(iter(uploaded))
except Exception:
    path = "your_image.png"  # running outside Colab: set a local path
print("file:", path)

## 4. Detect

In [ ]:
det = synthscope.load("cure")
result = det.predict(path)
print(f"Verdict: {result['verdict'].upper()}  |  P(AI-generated) = {result['fake_probability']:.3f}")
result

## Next steps

- Add more detectors: put their weights in the `CKPT` folder as `<key>.pth` (see each model's README), then `synthscope.load(key)`.
- Training-free video detectors need no weights: `synthscope.load('d3').predict('clip.mp4')`.
- Use your own trained checkpoint: `synthscope.load('cnn', checkpoint='/path/to.pth')`.
- See the [repo](https://github.com/SMSD75/SynthScope) for the web demo, the benchmarks (datasets download themselves), and how to add a detector.